<p align="center">
  <span style="color:Navy; font-size:200%; font-weight:bold; vertical-align:middle;">
        Estadística y Probabilidad    
  </span>
</p>
<p align="center" style="line-height:1.2;">
  <span style="color:RoyalBlue; font-size:160%;">Tema 5: Comparaciones múltiples  </span><br/>
  <span style="color:DodgerBlue; font-size:140%;"> Pruebas de hipótesis </span><br/>
  <span style="font-size:100%;color:forestgreen"> Escuela Nacional de Ciencias de la Tierra  |  Semestre 2027-I</span>
</p>

---

# **<font color="Navy">  Laboratorio 7: Pruebas paramétricas y no paramétricas </font>**

###  Descripción del problema
(... y un poco de la base de datos...)


El maíz es uno de los cultivos más importantes a nivel mundial y su producción puede verse afectada por diferentes factores biológicos y ambientales. Uno de los problemas más comunes en los cultivos de maíz es la infestación, lo que puede comprometer la calidad y el rendimiento de las cosechas.

Para abordar este desafío, se han realizado diversas campañas de muestreo y
reconocimiento en los años 2022 y 2023, en cultivos de maíz del Estado de San Luis Potosí por parte del Laboratorio Nacional de Geo-procesamiento de Información Fitosanitaria LaNGIF.

Durante estas campañas, se recolectaron múltiples variables, como el área del predio, la etapa fenológica de las plantas, el porcentaje de infestación por gusano Cogollero (Helicoverpa armígera), y otros factores importantes que permiten analizar la relación entre las características del cultivo y el nivel de infestación.

Este ejercicio se centrará en analizar el porcentaje de infestación en función de variables categóricas usando inicialmente la variedad de maíz como factor de comparación.

Utilizaremos un análisis de varianza (ANOVA) de una vía para comparar las medias de infestación entre las variedades de maíz.

Al finalizar esta actividad deben reponder las siguientes preguntas:
1. ¿Existe una diferencia significativa en el porcentaje de infestación entre las
diferentes variedades de maíz?
2. ¿Qué otras variables categóricas podrían influir significativamente en el porcentaje
de infestación?

In [ ]:
## Correr alguna de las siguientes líneas si no están instaladas las librerías.
%pip install --user scikit_posthocs pingouin statsmodels

In [ ]:
## Cargamos las librerías
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import levene, kruskal
import scikit_posthocs as sp
import pingouin as pg

In [ ]:
df = pd.read_csv('Maiz_22-23_v.csv', low_memory=False)
df.head()

In [ ]:
# Verificar los nombres de las columnas para entender mejor la estructura de los datos
df.columns

#### Elección de variables y visualización de los datos

In [ ]:
# Seleccionar las columnas de interés para el ANOVA
datos_anova = df[['Infestacion_%', 'Variedad']].rename(columns={'Infestacion_%': 'Infestacion'})
datos_anova.head()

In [ ]:
#Ahora averiguemos qué valores toma el porcentaje de infestación
datos_anova['Infestacion'].unique()

In [ ]:
#Y veamos cuáles son las variedad de maiz que existen
datos_anova['Variedad'].unique()

In [ ]:
# Visualización de los resultados con un boxplot
sns.boxplot(x='Variedad', y='Infestacion', data=datos_anova)
plt.xticks(rotation=90)
plt.title('Distribución de % de Infestación por Variedad')
plt.xlabel('Variedad')
plt.ylabel('% de Infestación')
plt.show()

# Implementación del ANOVA

In [ ]:
# Realizar el ANOVA
#crea un modelo de ANOVA usando la fórmula Infestacion ~ C(Variedad)
#con la función ols (Ordinary Least Squares) de statsmodels
# 'Infestacion ~ C(Variedad)': Especifica el modelo en fórmula,
#donde Infestacion es la variable dependiente y Variedad es la variable independiente.
# La C() alrededor de Variedad indica que Variedad es una variable categórica.
# data=datos_anova: Usa el DataFrame datos_anova que contiene las columnas necesarias.
#.fit(): Ajusta el modelo a los datos.

modelo = ols('Infestacion ~ C(Variedad)', data=datos_anova).fit()
resultado_anova = sm.stats.anova_lm(modelo, typ=2)
# Resumen del ANOVA
resultado_anova

Un valor F más alto sugiere que las diferencias entre las variedades son mucho más grandes que la variabilidad dentro de los grupos de variedades, lo que indica que las diferencias entre las variedades son estadísticamente significativas.

Valor p (Pr(>F)):p < 2.40e-40: Este valor p tan pequeño indica que hay una diferencia significativa entre las medias de infestación de al menos una de las variedades de maíz. La probabilidad de que estas diferencias se deban al azar es prácticamente nula.

In [ ]:
#Otra forma de implementar el ANOVA
from scipy.stats import f_oneway

# Datos
infestacion = datos_anova['Infestacion']
variedad = datos_anova['Variedad']

# ANOVA de una vía
resultado = f_oneway(*[infestacion[variedad == v] for v in variedad.unique()])
print(resultado)

In [ ]:
# Y una más
# ANOVA de una vía
resultado = pg.anova(data=datos_anova, dv='Infestacion', between='Variedad', detailed=True)
print(resultado)

**statsmodels con ols**
**Ventajas:** statsmodels proporciona una interfaz poderosa y muy flexible para realizar análisis estadísticos. Usar ols te permite construir modelos con gran detalle, realizar diagnósticos y obtener interpretaciones detalladas de los resultados, incluyendo análisis de residuos y gráficos de diagnóstico.

**Desventajas:** Puede ser un poco más complejo de usar para tareas simples y requiere una buena comprensión de las fórmulas y la teoría detrás del modelo.

**scipy con f_oneway**
**Ventajas:** scipy es una biblioteca muy ligera y rápida para realizar análisis estadísticos. f_oneway es fácil de usar y adecuada para realizar ANOVA de una vía rápidamente.

**Desventajas:** Ofrece menos funcionalidad para el diagnóstico y la interpretación de los resultados comparado con statsmodels. No proporciona pruebas post-hoc integradas, por lo que tendrías que realizarlas por separado si es necesario.

**pingouin**
**Ventajas:** pingouin es una biblioteca relativamente nueva y orientada a la simplicidad y la facilidad de uso. Ofrece una variedad de pruebas estadísticas y métodos post-hoc integrados. El resultado es detallado y fácil de interpretar.

**Desventajas:** Puede no ser tan conocida o utilizada en la comunidad estadística, lo que puede llevar a menos documentación y ejemplos disponibles.

# Prueba post-hoc

# ¿Paramétrica o no paramética?

Para saber cuál tipo de prueba usar, primero usamos **la prueba de Levene** para saber si la varianza entre los grupos es la misma o no.

¿Cuál es la hipótesis nula y la alternativa?
¿Cómo debe ser el valor p para cada una?


In [ ]:
# Convierte la columna 'Variedad' a numérica si es necesario
# Asumiendo que 'Variedad' es categórica, la convertimos a códigos numéricos
datos_anova['Variedad_Coded'] = pd.factorize(datos_anova['Variedad'])[0]

# Ahora usa la columna codificada en la prueba de Levene
stat, p = levene(datos_anova['Infestacion'], datos_anova['Variedad_Coded'])
print(f'Levene test: stat={stat}, p={p}')

**La prueba de Turkey** nos dará información sobre cuáles son los grupos que tienen diferencias significativas entre ellos.

H0: Los 2 grupos tienen medias iguales

H1: Los 2 grupos tienen medias diferentes

In [ ]:
# Si el ANOVA es significativo, hacer la prueba post-hoc de Tukey
if resultado_anova["PR(>F)"][0] < 0.05:
    print(sm.stats.multicomp.pairwise_tukeyhsd(datos_anova['Infestacion'], datos_anova['Variedad']))


¿Qué nos dicen estos resultados?

Recordemos que el objetivo era implementar el ANOVA de una vía para comparar las medias de infestación entre las variedades de maíz.

¿Existe una diferencia significativa en el porcentaje de infestación entre las diferentes variedades de maíz?

# Verificación de supuestos

Ya:
- Aplicamos el ANOVA de una vía.
- Hicimos la prueba de Leviene para saber si la varianza de los grupos es la misma o no.
- Realizamos una prueba post-hoc para saber cuáles grupos son estadísticamente similares entre ellos.

Pero ¿Són válidos nuestros resultados?

Recordemos que, para que el ANOVA sea válido, se deben de cumplir algunos supuestos:

1. Independencia: Las observaciones dentro de cada grupo y entre los grupos deben ser independientes.
2. Normalidad: Los datos dentro de cada grupo deben seguir aproximadamente una distribución normal.
3. Homogeneidad de varianzas (Homocedasticidad): Las varianzas entre los grupos deben ser similares.

Veamos si se cumplen estos supuestos.

In [ ]:
# Ya habíamos hecho este paso, pero pongámos más atención.
# ¿Hay normalidad dentro de cada grupo?

# Visualización de los resultados con un boxplot
sns.boxplot(x='Variedad', y='Infestacion', data=datos_anova)
plt.xticks(rotation=90)
plt.title('Distribución de % de Infestación por Variedad')
plt.xlabel('Variedad')
plt.ylabel('% de Infestación')
plt.show()

In [ ]:
# Aún con esa ayuda visual, debemos implementar otros recursos
# para justificar nuestras respuestas

# 1. Verificación de la normalidad de los residuos
residuos = modelo.resid

# Histograma de los residuos
#Idealmente, deberían seguir una distribución normal.
plt.hist(residuos, bins=30)
plt.title('Histograma de los Residuos')
plt.xlabel('Residuos')
plt.show()

# ¿Qué nos dicen estos resultados?
# ¿Hay normalidad en los residuos?
# ¿Se cumple el punto 2?

In [ ]:
#Este es uno muy común, por ejemplo.

# Gráfico Q-Q para verificar normalidad
# El gráfico Q-Q se usa para evaluar si los residuos siguen una distribución normal.
# Los puntos deberían alinearse a lo largo de una línea recta.
sm.qqplot(residuos, line='s')
plt.show()

Si los supuestos no se cumplen, se pueden realizar transformaciones a los datos.
Sin embargo, El ANOVA es robusto ante desviaciones moderadas de la normalidad, especialmente si tenemos tamaños de muestra grandes.
Una alternativa es usar una prueba no paramétrica, como la prueba de Kruskal-Wallis, que no asume normalidad.

---

Ya notamos que no se cumplen con los supuestos del ANOVA. Así que continuaremos con la implementación de la prueba de Kruskal-Wallis, que es una prueba no paramétrica.

**¿Qué diferencias se identifican usando la prueba de Kruskal-Wallis en comparación con los resultados del ANOVA?**

---

# Implementación de la prueba Kruskal-Wallis

La prueba de **Kruskal-Wallis** es un método no paramétrico para comprobar si las muestras proceden de la misma distribución. Comprueba si los rangos medios de los grupos son iguales.

*   **H0**: Las muestras independientes tienen todas la misma tendencia central y, por tanto, proceden de la misma población.
*   **H1**: Al menos una de las muestras independientes no tiene la misma tendencia central que las demás muestras y, por tanto, procede de una población diferente.



In [ ]:
# Asumiendo que 'Variedad' es categórica, la convertimos a códigos numéricos
datos_anova['Variedad_Coded'] = pd.factorize(datos_anova['Variedad'])[0]

#Realizamos la prueba de Kruskal-Wallis
kruskal_stat, kruskal_p = kruskal(datos_anova['Infestacion'], datos_anova['Variedad_Coded'])
print(f'Kruskal-Wallis test: stat={kruskal_stat}, p={kruskal_p}')

Un valor H más alto sugiere que las diferencias entre las medias de los rangos son estadísticamente significativas.

Este valor p tan pequeño indica que hay una diferencia significativa entre las medias de los rangos del porcentaje de infestación de al menos una de las variedades de maíz.

En este caso **rechazamos la hipótesis nula**, es decir los porcentajes de infestación no son los mismos para cada tipo de maiz.  **Pero** aún no sabemos cuáles tipos de maiz tienen porcentajes diferentes.

Acontinuación, implementamos la prueba Post-hoc no paramétrica de Dunn para saber cuáles grupos son diferentes entre sí.

# Prueba post-hoc

**La prueba de Dunn** nos dará información sobre cuáles son los grupos que tienen diferencias significativas entre ellos.

H0: Los 2 grupos tienen medias de rangos iguales

H1: Los 2 grupos tienen medias de rangos diferentes


In [ ]:
# Prueba de Dunn post-hoc
dunn = sp.posthoc_dunn(datos_anova, val_col='Infestacion', group_col='Variedad', p_adjust='bonferroni')

## Acontinuación veremos los p-values obtenidos de la prueba de Dunn
# ¿Cuáles grupos tienen porcentajes de infestación diferentes?
print(dunn)

In [ ]:
#Comparamos la Prueba de Dunn con los resultados de la prueba de Turkey
print(sm.stats.multicomp.pairwise_tukeyhsd(datos_anova['Infestacion'], datos_anova['Variedad']))

* ¿Existe una diferencia significativa en el porcentaje de infestación entre las diferentes variedades de maíz?
* ¿Entre cuáles?
* ¿Qué diferencias se identifican usando la prueba de Kruskal-Wallis en comparación con los resultados del ANOVA?

---
<a name='ej-1'></a>
### **<font color="DodgerBlue"> Ejercicio :   </font>**

<font color="DarkBlue"> **¿Qué otras variables categóricas podrían influir significativamente en el porcentaje de infestación?**
Su misión ahora es elegir otra variable categórica (que tenga sentido), replicar e interpretar lo que se realizó en esta clase.

Manden su notebook, en este deben discutir sus resultados.

---